# Fine-tuning Qwen2.5-1.5B-Instruct on arXiv Papers (Abstract → Title)

Fine-tuning `Qwen/Qwen2.5-1.5B-Instruct` with QLoRA on a subset of the ML-ArXiv-Papers dataset.

## Step 1: Install libraries and imports

In [1]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(torch.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
2.11.0+cu130
True Tesla T4


## Step 2: Load dataset
We use the ML-ArXiv-Papers dataset (title + abstract). A random subset of 5,000 samples is used for training.

In [2]:
ds = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
print(ds)

# keep a random 5k subset
ds = ds.shuffle(seed=42).select(range(5000))
print(len(ds))

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv: reconstructing file:   0%|          |  0.00B /  147MB            

ML-Arxiv-Papers.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})
5000


## Step 3: Format data for Qwen
Clean the abstracts, wrap each pair in Qwen's chat format (abstract as the user turn, title as the assistant turn), and split into train and validation sets.

In [3]:
def clean(text):
    return " ".join(text.split())

def to_chat(row):
    abstract = clean(row["abstract"])
    title = clean(row["title"])
    text = (
        f"<|im_start|>user\nWrite a title for this abstract:\n{abstract}<|im_end|>\n"
        f"<|im_start|>assistant\n{title}<|im_end|>"
    )
    return {"text": text}

ds = ds.map(to_chat, remove_columns=ds.column_names)

split = ds.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(len(train_ds), len(val_ds))
print(train_ds[0]["text"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

4750 250
<|im_start|>user
Write a title for this abstract:
We propose a novel memory-based tracker via part-level dense memory and voting-based retrieval, called DMV. Since deep learning techniques have been introduced to the tracking field, Siamese trackers have attracted many researchers due to the balance between speed and accuracy. However, most of them are based on a single template matching, which limits the performance as it restricts the accessible in-formation to the initial target features. In this paper, we relieve this limitation by maintaining an external memory that saves the tracking record. Part-level retrieval from the memory also liberates the information from the template and allows our tracker to better handle the challenges such as appearance changes and occlusions. By updating the memory during tracking, the representative power for the target object can be enhanced without online learning. We also propose a novel voting mechanism for the memory reading to filter 

## Step 4: Load model and tokenizer (4-bit)
Load `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit (NF4) so it fits comfortably in GPU memory.

In [4]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(model.get_memory_footprint() / 1e9, "GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

1.589172736 GB


## Step 5: LoRA setup
Attach LoRA adapters to the attention and MLP layers. Only these small adapter weights are trained, while the 4-bit base model stays frozen.

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Step 6: Training arguments and trainer
Training settings (2 epochs, effective batch size 16, cosine learning rate schedule) and the SFTTrainer that ties the model, data and settings together.

In [8]:
training_args = SFTConfig(
    output_dir="./qwen-arxiv-title",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

# only one GPU is used for training
training_args._n_gpu = 1

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print(len(train_ds) // (4 * 4), "steps per epoch")

296 steps per epoch


In [10]:
# trainable weights must be float32 for fp16 training
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

print({p.dtype for p in model.parameters() if p.requires_grad})

{torch.float32}


## Step 7: Train
Run training. Loss is logged every 20 steps and validation loss every 100 steps.

In [11]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,2.291940,2.333441,2.332812,0.502711,392033.000000
200,2.305191,2.327706,2.315962,0.504250,774136.000000
300,2.310043,2.322695,2.302280,0.505360,1162582.000000
400,2.258856,2.329294,2.253275,0.502914,1548595.000000
500,2.215973,2.328674,2.247798,0.504122,1938141.000000
594,2.205038,2.328207,2.246531,0.504213,2297901.000000


TrainOutput(global_step=594, training_loss=2.281176862491903, metrics={'train_runtime': 3416.6236, 'train_samples_per_second': 2.781, 'train_steps_per_second': 0.174, 'total_flos': 2.373202816719667e+16, 'train_loss': 2.281176862491903, 'epoch': 2.0})

## Step 8: Test the model
Generate titles for a few validation abstracts and compare them with the original titles.

In [12]:
model.config.use_cache = True
model.eval()

marker = "<|im_start|>assistant\n"

for i in range(5):
    text = val_ds[i]["text"]
    prompt = text.split(marker)[0] + marker
    actual = text.split(marker)[1].replace("<|im_end|>", "")

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    pred = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("Actual   :", actual)
    print("Generated:", pred.strip())
    print()

Actual   : Machine learning structure preserving brackets for forecasting irreversible processes
Generated: Learning Dissipative Systems with Metriplectic Brackets

Actual   : Foveation for Segmentation of Ultra-High Resolution Images
Generated: Foveation: A Data Loader for Ultra-High Resolution Image Segmentation

Actual   : Data Interpolating Prediction: Alternative Interpretation of Mixup
Generated: Data Interpolating Prediction: A Framework for Learning with Augmented Samples

Actual   : Grammar Filtering For Syntax-Guided Synthesis
Generated: Learning to Solve Programming-by-Example Problems

Actual   : Loss-Aversively Fair Classification
Generated: Loss-Averse Updates: A Behavioral Perspective on Fairness in Algorithmic Decision Making



## Step 9: Save adapter
Save the trained LoRA adapter and tokenizer so the model can be reloaded later.

In [13]:
save_dir = "./qwen-arxiv-title-adapter"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

!ls -lh {save_dir}

total 82M
-rw-r--r-- 1 root root 1.2K Nov 13 07:38 adapter_config.json
-rw------- 1 root root  71M Nov 13 07:38 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Nov 13 07:38 chat_template.jinja
-rw-r--r-- 1 root root 5.1K Nov 13 07:38 README.md
-rw-r--r-- 1 root root  694 Nov 13 07:38 tokenizer_config.json
-rw-r--r-- 1 root root  11M Nov 13 07:38 tokenizer.json
-rw-r--r-- 1 root root 5.6K Nov 13 07:38 training_args.bin


## Step 10: Evaluate with ROUGE
Generate titles for all 250 validation abstracts and compare them with the original titles using ROUGE-1, ROUGE-2 and ROUGE-L.

In [14]:
!pip install -q evaluate rouge_score

import evaluate

rouge = evaluate.load("rouge")

model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"

marker = "<|im_start|>assistant\n"
prompts, actuals = [], []
for row in val_ds:
    parts = row["text"].split(marker)
    prompts.append(parts[0] + marker)
    actuals.append(parts[1].replace("<|im_end|>", "").strip())

preds = []
batch_size = 16
for i in range(0, len(prompts), batch_size):
    batch = prompts[i:i + batch_size]
    inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda:0")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[:, inputs["input_ids"].shape[1]:]
    preds += [t.strip() for t in tokenizer.batch_decode(gen, skip_special_tokens=True)]

scores = rouge.compute(predictions=preds, references=actuals)
print({k: round(v, 4) for k, v in scores.items()})

tokenizer.padding_side = "right"

{'rouge1': np.float64(0.4477), 'rouge2': np.float64(0.249), 'rougeL': np.float64(0.399), 'rougeLsum': np.float64(0.3998)}


## Step 11: Baseline comparison
Score the same validation set with the LoRA adapter disabled, to measure how much the fine-tuning improved the base model.

In [15]:
tokenizer.padding_side = "left"

base_preds = []
with model.disable_adapter():
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda:0")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        gen = out[:, inputs["input_ids"].shape[1]:]
        base_preds += [t.strip() for t in tokenizer.batch_decode(gen, skip_special_tokens=True)]

base_scores = rouge.compute(predictions=base_preds, references=actuals)

print("Base model :", {k: round(float(v), 4) for k, v in base_scores.items()})
print("Fine-tuned :", {k: round(float(v), 4) for k, v in scores.items()})

tokenizer.padding_side = "right"

Base model : {'rouge1': 0.4053, 'rouge2': 0.2122, 'rougeL': 0.348, 'rougeLsum': 0.349}
Fine-tuned : {'rouge1': 0.4477, 'rouge2': 0.249, 'rougeL': 0.399, 'rougeLsum': 0.3998}
